# Import Dependencies

In [ ]:
import pandas as pd
import numpy as np
import shap
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Dataset Load

In [ ]:
shift_train_df = pd.read_csv("datasets/Attack/ciciomt2024_shift_train_seen_attacks.csv")
shift_val_df = pd.read_csv("datasets/Attack/ciciomt2024_shift_val_seen_attacks.csv")
shift_test_df = pd.read_csv("datasets/Attack/ciciomt2024_shift_test_unseen_attacks.csv")

# XGBoost Training

In [ ]:
non_feature_cols = [
    "source_file",
    "attack_type",
    "attack_family",
    "binary_label"
]

feature_cols_shift = [
    col for col in shift_train_df.columns
    if col not in non_feature_cols
]

X_shift_train = shift_train_df[feature_cols_shift]
y_shift_train = shift_train_df["binary_label"]

X_shift_val = shift_val_df[feature_cols_shift]
y_shift_val = shift_val_df["binary_label"]

X_shift_test = shift_test_df[feature_cols_shift]
y_shift_test = shift_test_df["binary_label"]

xgb_shift = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_shift.fit(X_shift_train, y_shift_train)

# Family-Balanced Unseen Attack-Family Shift Test

In [ ]:
# ============================================================
# Purpose:
# Create equal unseen attack-family samples:
# Recon = 1744, ARP = 1744, Malformed = 1744
# Benign = 5232
# Then evaluate the already-trained xgb_shift model.
# ============================================================

# -------------------------------
# 1. Check required variables
# -------------------------------
print("Checking available data...")
print("shift_test_df shape:", shift_test_df.shape)
print("Feature count:", len(feature_cols_shift))

print("\nOriginal shift test attack_family distribution:")
print(shift_test_df["attack_family"].value_counts())

print("\nOriginal shift test label distribution:")
print(shift_test_df["binary_label"].value_counts())

# -------------------------------
# 2. Separate families
# -------------------------------
benign_df = shift_test_df[shift_test_df["attack_family"] == "Benign"]
recon_df = shift_test_df[shift_test_df["attack_family"] == "Recon"]
arp_df = shift_test_df[shift_test_df["attack_family"] == "ARP"]
malformed_df = shift_test_df[shift_test_df["attack_family"] == "Malformed"]

print("\nAvailable samples:")
print("Benign:", len(benign_df))
print("Recon:", len(recon_df))
print("ARP:", len(arp_df))
print("Malformed:", len(malformed_df))

# -------------------------------
# 3. Create family-balanced attack subset
# -------------------------------
# Minimum available unseen family size
n_per_attack_family = min(len(recon_df), len(arp_df), len(malformed_df))

print("\nSamples per unseen attack family:", n_per_attack_family)

recon_sample = recon_df.sample(n=n_per_attack_family, random_state=42)
arp_sample = arp_df.sample(n=n_per_attack_family, random_state=42)
malformed_sample = malformed_df.sample(n=n_per_attack_family, random_state=42)

balanced_attack_df = pd.concat([
    recon_sample,
    arp_sample,
    malformed_sample
])

# Binary-balanced benign sample: same number as total attacks
n_benign = len(balanced_attack_df)

benign_sample = benign_df.sample(n=n_benign, random_state=42)

family_balanced_shift_test_df = pd.concat([
    benign_sample,
    balanced_attack_df
])

family_balanced_shift_test_df = family_balanced_shift_test_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nFamily-balanced shift test shape:", family_balanced_shift_test_df.shape)

print("\nFamily-balanced label distribution:")
print(family_balanced_shift_test_df["binary_label"].value_counts())

print("\nFamily-balanced attack_family distribution:")
print(family_balanced_shift_test_df["attack_family"].value_counts())

# -------------------------------
# 4. Create X and y
# -------------------------------
X_family_balanced = family_balanced_shift_test_df[feature_cols_shift]
y_family_balanced = family_balanced_shift_test_df["binary_label"]

# -------------------------------
# 5. Evaluate xgb_shift model
# -------------------------------
y_family_bal_pred = xgb_shift.predict(X_family_balanced)
y_family_bal_prob = xgb_shift.predict_proba(X_family_balanced)[:, 1]

accuracy = accuracy_score(y_family_balanced, y_family_bal_pred)
precision = precision_score(y_family_balanced, y_family_bal_pred)
recall = recall_score(y_family_balanced, y_family_bal_pred)
f1 = f1_score(y_family_balanced, y_family_bal_pred)
roc_auc = roc_auc_score(y_family_balanced, y_family_bal_prob)
cm = confusion_matrix(y_family_balanced, y_family_bal_pred)

print("\nFamily-Balanced Unseen Attack-Family Shift Test Metrics:")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC-AUC:", roc_auc)
print("Confusion Matrix:")
print(cm)

# -------------------------------
# 6. Family-wise recall
# -------------------------------
family_balanced_shift_test_df["predicted_label"] = y_family_bal_pred

family_recall_rows = []

for family in ["Recon", "ARP", "Malformed"]:
    family_df = family_balanced_shift_test_df[
        family_balanced_shift_test_df["attack_family"] == family
    ]

    total_attacks = (family_df["binary_label"] == 1).sum()
    correctly_detected = (
        (family_df["binary_label"] == 1) &
        (family_df["predicted_label"] == 1)
    ).sum()

    family_recall = correctly_detected / total_attacks if total_attacks > 0 else 0

    family_recall_rows.append({
        "attack_family": family,
        "correctly_detected": correctly_detected,
        "total_attacks": total_attacks,
        "recall": family_recall
    })

family_balanced_recall_df = pd.DataFrame(family_recall_rows)

print("\nFamily-wise recall on family-balanced unseen shift test:")
print(family_balanced_recall_df)